# DataLeague Datathon - Manipulation and Anomaly Detection

This notebook is the jury-facing runner. The reusable implementation lives in `datathon_pipeline.py`.

In [ ]:
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ["duckdb", "matplotlib"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", missing)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Install failed. If you see externally-managed-environment, create a venv and select .venv/bin/python as this notebook kernel. See README.md.") from exc
else:
    print("Runtime dependencies are ready.")

In [ ]:
from pathlib import Path
import importlib
import pandas as pd

import datathon_pipeline as dp
dp = importlib.reload(dp)

DATA_PATH = dp.DATA_PATH
ARTIFACT_DIR = dp.ARTIFACT_DIR
inspect_dataset = dp.inspect_dataset
run_pipeline = dp.run_pipeline
predict_live = dp.predict_live
load_context = dp.load_context

print("Pipeline version:", dp.MODEL_VERSION)

DATA_PATH, ARTIFACT_DIR

## 1. Dataset Check

In [ ]:
info = inspect_dataset(DATA_PATH)
print("Rows:", f"{info['row_count']:,}")
display(info["schema"])
display(info["date_range"])

## 2. Score Sample and Build Artifacts

Increase `SAMPLE_SIZE` if you have time. Use 20k-50k for a fast demo, 200k+ for the final run.

In [ ]:
SAMPLE_SIZE = 200_000

scored, context, summary = run_pipeline(
    data_path=DATA_PATH,
    artifacts_dir=ARTIFACT_DIR,
    sample_size=SAMPLE_SIZE,
    use_full_author_stats=True,
    make_plots=True,
)
summary

## 3. Highest Risk Non-Empty Examples and Empty Text Anomalies

In [ ]:
cols = [
    "risk_score", "risk_band", "organic_score", "label", "reason_codes",
    "content_risk", "author_risk", "coordination_risk",
    "language", "platform_domain", "primary_theme", "author_hash",
    "cluster_size", "cluster_author_nunique", "cluster_confidence_score",
    "stripped_char_len", "original_text"
]
existing = [c for c in cols if c in scored.columns]

print("Empty text anomalies are treated as high risk, but separated from content examples.")
empty_summary = (
    scored[scored["reason_codes"].fillna("").str.contains("EMPTY_TEXT")]
    .groupby(["platform_domain", "language"], dropna=False)
    .size()
    .reset_index(name="empty_text_posts")
    .sort_values("empty_text_posts", ascending=False)
)
display(empty_summary.head(10))

print("Highest risk examples with real text content")
non_empty_examples = scored[scored["stripped_char_len"].fillna(0) >= 20].copy()
display(
    non_empty_examples
    .sort_values(["risk_score", "coordination_risk", "author_risk"], ascending=False)
    [existing]
    .head(20)
)

## 4. Manipulation Map Artifacts

In [ ]:
from IPython.display import Image, display

for image_name in [
    "risk_score_histogram.png",
    "risk_map_language_platform.png",
    "language_manipulative_share.png",
    "top_suspicious_segments.png",
    "platform_normalized_risk.png",
    "top_risk_authors.png",
    "hourly_suspicious_share.png",
]:
    path = ARTIFACT_DIR / image_name
    if path.exists():
        display(Image(filename=str(path)))

display(pd.read_csv(ARTIFACT_DIR / "top_risk_segments.csv").head(10))
display(pd.read_csv(ARTIFACT_DIR / "platform_normalized_risk.csv").head(10))
display(pd.read_csv(ARTIFACT_DIR / "top_risk_authors.csv").head(10))
display(pd.read_csv(ARTIFACT_DIR / "top_coordination_clusters.csv").head(10))

## 5. Marketing Evidence

In [ ]:
from IPython.display import Image, Markdown, display

scorecard_path = ARTIFACT_DIR / "marketing_scorecard.md"
if scorecard_path.exists():
    display(Markdown(scorecard_path.read_text(encoding="utf-8")))

for image_name in [
    "risk_funnel.png",
    "reason_code_breakdown.png",
    "psychological_trigger_breakdown.png",
    "live_inference_benchmark.png",
    "coordination_confidence_bubble.png",
    "evidence_quality_summary.png",
]:
    path = ARTIFACT_DIR / image_name
    if path.exists():
        display(Image(filename=str(path)))

for table_name in ["live_inference_benchmark.csv", "presentation_examples.csv"]:
    path = ARTIFACT_DIR / table_name
    if path.exists():
        print(table_name)
        display(pd.read_csv(path).head(10))

## 6. Live Inference Demo

In [ ]:
import importlib
import datathon_pipeline as dp
dp = importlib.reload(dp)
predict_live = dp.predict_live
load_context = dp.load_context
context = load_context(ARTIFACT_DIR)
print("Pipeline version:", dp.MODEL_VERSION)
print("Loaded fresh inference context from artifacts.")

demo_texts = [
    "The city council published meeting notes and invited residents to comment.",
    "BUY NOW!!! FREE FREE FREE #deal #promo https://spam.example.com",
    "",
]

for text in demo_texts:
    print("TEXT:", repr(text))
    print(predict_live(text, context=context))
    print("-" * 80)

## 7. Jury Input Cell

Paste the hidden test text below during the presentation.

In [ ]:
import importlib
import datathon_pipeline as dp
dp = importlib.reload(dp)
predict_live = dp.predict_live
load_context = dp.load_context
context = load_context(ARTIFACT_DIR)
print("Pipeline version:", dp.MODEL_VERSION)
print("Loaded fresh inference context from artifacts.")

hidden_text = "PASTE JURY TEXT HERE"
predict_live(hidden_text, context=context)

## 8. Jury CSV Batch Inference

Put the CSV file from the jury into `jury_inputs/`. In the next cell, change only `JURY_CSV_FILENAME` to the file name. The predictions will be written under `artifacts/`.

In [ ]:
import importlib
from pathlib import Path
import pandas as pd

import datathon_pipeline as dp
dp = importlib.reload(dp)

# 1) Put the jury CSV under this folder.
JURY_INPUT_DIR = Path("jury_inputs")

# 2) During the presentation, change only this file name.
JURY_CSV_FILENAME = "jury_test.csv"

# Usually leave these defaults as-is.
TEXT_COLUMN = None          # None auto-detects original_text/text or scores all non-metadata columns.
CSV_HAS_HEADER = True       # Set False if the jury CSV has no header row.
CSV_SEPARATOR = ","         # Use ";" for semicolon CSV or "auto" for delimiter sniffing.
USE_ALL_CELLS = False       # Set True if every non-metadata cell may contain a text.

JURY_INPUT_DIR.mkdir(exist_ok=True)
jury_csv_path = JURY_INPUT_DIR / JURY_CSV_FILENAME
if not jury_csv_path.exists():
    raise FileNotFoundError(
        f"CSV not found: {jury_csv_path.resolve()}\n"
        "Put the jury file under jury_inputs/ and update JURY_CSV_FILENAME."
    )

output_path = ARTIFACT_DIR / f"jury_predictions_{jury_csv_path.stem}.csv"
effective_text_column = None if USE_ALL_CELLS else TEXT_COLUMN

predictions, jury_summary = dp.predict_csv_file(
    input_csv=jury_csv_path,
    output_csv=output_path,
    text_column=effective_text_column,
    all_cells=USE_ALL_CELLS,
    no_header=not CSV_HAS_HEADER,
    csv_sep=CSV_SEPARATOR,
    artifacts_dir=ARTIFACT_DIR,
)

print("Input CSV:", jury_csv_path)
print("Output CSV:", output_path)
print("Rows/cells scored:", jury_summary["texts_scored"])
print("Pipeline version:", dp.MODEL_VERSION)

band_summary = (
    predictions["risk_band"]
    .value_counts(dropna=False)
    .rename_axis("risk_band")
    .reset_index(name="count")
)
band_summary["share"] = (band_summary["count"] / max(len(predictions), 1)).round(4)
display(band_summary)

preview_cols = [
    "source_row", "source_column", "risk_score", "risk_band", "label",
    "evidence_level", "top_reasons", "psychological_triggers", "original_text"
]
display(predictions[[c for c in preview_cols if c in predictions.columns]].head(30))
